# Fed, Pieniądz i Rynki — Dzień 3: Poprawki + Dashboard

**Plan:**
1. Naprawa SP500 — pełna historia od 1990 przez yfinance
2. Naprawa Q2 — inwersja → recesja (Python zamiast SQL)
3. Interaktywny dashboard HTML (Plotly) do portfolio

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH  = Path('../data/fed_cycles.db')
RAW_DIR  = Path('../data/raw')
REP_DIR  = Path('../reports')

conn = sqlite3.connect(DB_PATH)
print('Połączono z bazą:', DB_PATH.name)

Połączono z bazą: fed_cycles.db


---
## NAPRAWA 1 — SP500 pełna historia od 1990

FRED bez klucza daje SP500 tylko od ~2016. Pobieramy pełną historię przez yfinance (^GSPC),
agregujemy do miesięcznej średniej i aktualizujemy bazę.

In [2]:
import yfinance as yf

print('Pobieranie S&P500 (^GSPC) z Yahoo Finance...')
gspc = yf.download('^GSPC', start='1990-01-01', progress=False)

# Średnia miesięczna (pierwszego dnia miesiąca)
sp_monthly = (
    gspc['Close']
    .resample('MS').mean()
    .reset_index()
)
sp_monthly.columns = ['date', 'value']
sp_monthly['date']      = sp_monthly['date'].dt.strftime('%Y-%m-%d')
sp_monthly['series_id'] = 'SP500'

print(f'Pobrano: {len(sp_monthly)} miesięcy ({sp_monthly["date"].iloc[0]} → {sp_monthly["date"].iloc[-1]})')
print(f'Min: {sp_monthly["value"].min():.0f}  Max: {sp_monthly["value"].max():.0f}')

Pobieranie S&P500 (^GSPC) z Yahoo Finance...


Pobrano: 441 miesięcy (1990-01-01 → 2026-09-01)
Min: 307  Max: 7711


In [3]:
# Zapisz do CSV (backup)
sp_monthly.to_csv(RAW_DIR / 'SP500_full.csv', index=False)
print('Zapisano: data/raw/SP500_full.csv')

# Zastąp w bazie: usuń stary SP500, wstaw nowy
conn.execute("DELETE FROM raw_series WHERE series_id = 'SP500'")
sp_monthly.to_sql('raw_series', conn, if_exists='append', index=False)
conn.commit()

# Weryfikacja
n = conn.execute("SELECT COUNT(*) FROM raw_series WHERE series_id='SP500'").fetchone()[0]
print(f'SP500 w bazie: {n} wierszy')

Zapisano: data/raw/SP500_full.csv
SP500 w bazie: 441 wierszy


---
## NAPRAWA 2 — Q2: Inwersja krzywej → recesja (logika w Pythonie)

Poprzedni SQL był zbyt restrykcyjny. Liczymy to prościej:
dla każdej recesji → znajdź ostatnią inwersję PRZED jej startem → policz różnicę miesięcy.

In [4]:
# Wczytaj dane z v_master
df = pd.read_sql("""
    SELECT ym, t10y2y, usrec
    FROM v_master
    WHERE t10y2y IS NOT NULL AND usrec IS NOT NULL
    ORDER BY ym
""", conn)
df['ym'] = pd.to_datetime(df['ym'])

# Znajdź początki inwersji (przejście z >= 0 na < 0)
df['prev_t10y2y'] = df['t10y2y'].shift(1)
inwersje = df[(df['t10y2y'] < 0) & (df['prev_t10y2y'] >= 0)]['ym'].tolist()

# Znajdź początki recesji (przejście z 0 na 1)
df['prev_usrec'] = df['usrec'].shift(1)
recesje_start = df[(df['usrec'] == 1) & (df['prev_usrec'] == 0)]['ym'].tolist()

print(f'Znaleziono {len(inwersje)} inwersji krzywej:')
for d in inwersje:
    print(f'  {d.strftime("%Y-%m")}')

print(f'\nZnaleziono {len(recesje_start)} recesji:')
for d in recesje_start:
    print(f'  {d.strftime("%Y-%m")}')

Znaleziono 7 inwersji krzywej:
  1990-03
  1998-06
  2000-02
  2006-02
  2006-06
  2007-05
  2022-07

Znaleziono 4 recesji:
  1990-08
  2001-04
  2008-01
  2020-03


In [5]:
# Dla każdej recesji → znajdź poprzedzającą inwersję
nazwy_recesji = {0: 'Dot-com (2001)', 1: 'GFC (2007)', 2: 'COVID (2020)'}

print('\nInwersja krzywej → Recesja:')
print(f'{"Recesja":<20} {"Inwersja":>12} {"Start recesji":>15} {"Miesięcy"}')
print('-' * 60)

wyniki = []
for i, rec_start in enumerate(recesje_start):
    # ostatnia inwersja PRZED startem recesji (max 48 miesięcy wcześniej)
    poprzednie = [d for d in inwersje if d < rec_start and (rec_start - d).days < 48 * 30]
    if poprzednie:
        ostatnia_inwersja = max(poprzednie)
        miesiace = round((rec_start - ostatnia_inwersja).days / 30)
        nazwa = nazwy_recesji.get(i, f'Recesja {i+1}')
        print(f'{nazwa:<20} {ostatnia_inwersja.strftime("%Y-%m"):>12} {rec_start.strftime("%Y-%m"):>15} {miesiace:>8}')
        wyniki.append({'recesja': nazwa, 'inwersja': ostatnia_inwersja.strftime('%Y-%m'),
                       'start_recesji': rec_start.strftime('%Y-%m'), 'miesiecy': miesiace})
    else:
        nazwa = nazwy_recesji.get(i, f'Recesja {i+1}')
        print(f'{nazwa:<20} {"brak inwersji":>12}')

if wyniki:
    avg = round(sum(w['miesiecy'] for w in wyniki) / len(wyniki))
    print(f'\nŚredni czas inwersja → recesja: {avg} miesięcy')


Inwersja krzywej → Recesja:
Recesja                  Inwersja   Start recesji Miesięcy
------------------------------------------------------------
Dot-com (2001)            1990-03         1990-08        5
GFC (2007)                2000-02         2001-04       14
COVID (2020)              2007-05         2008-01        8
Recesja 4            brak inwersji

Średni czas inwersja → recesja: 9 miesięcy


## DASHBOARD — Interaktywny HTML (Plotly)

Osiem paneli w układzie 4×2:
1. S&P500 z zaznaczonymi recesjami
2. Yield curve + stopa Fed
3. M2 YoY
4. Inflacja CPI + realna stopa Fed
5. Bezrobocie (Sahm Rule)
6. VIX — indeks strachu
7. *(pusty)*
8. Recession Scorecard (gauge)

Plik wynikowy: `reports/dashboard.html`

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Wczytaj pełne dane
df = pd.read_sql('SELECT * FROM v_master ORDER BY ym', conn)
df['ym'] = pd.to_datetime(df['ym'])

# Wskaźniki pochodne
df['m2_yoy']    = df['m2sl'].pct_change(12) * 100
df['cpi_yoy']   = df['cpiaucsl'].pct_change(12) * 100
df['real_rate'] = df['fedfunds'] - df['cpi_yoy']
df['sp500_yoy'] = df['sp500'].pct_change(12) * 100

# Okresy recesji dla szarych pasów
recesje_df = pd.read_sql('SELECT * FROM dim_recession', conn)

print(f'Dane: {len(df)} miesięcy ({df["ym"].min().date()} → {df["ym"].max().date()})')

Dane: 441 miesięcy (1990-01-01 → 2026-09-01)


In [7]:
def dodaj_recesje(fig, row, col, recesje_df, y0=0, y1=1, yref_suffix=''):
    """Dodaje szare pasy recesji do wykresu Plotly."""
    yref = f'y{row}' if row > 1 else 'y'
    for _, r in recesje_df.iterrows():
        fig.add_vrect(
            x0=r['start'], x1=r['end'],
            fillcolor='rgba(150,150,150,0.15)',
            layer='below', line_width=0,
            annotation_text=r['name'],
            annotation_position='top left',
            annotation_font_size=9,
            annotation_font_color='#888',
            row=row, col=col
        )


# Kolory
C_BLUE   = '#1565C0'
C_RED    = '#C62828'
C_GREEN  = '#2E7D32'
C_ORANGE = '#E65100'
C_PURPLE = '#6A1B9A'
C_GRAY   = '#546E7A'

print('Funkcje pomocnicze gotowe.')

Funkcje pomocnicze gotowe.


In [8]:
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=(
        'S&P 500 — historia indeksu',
        'Yield Curve (10Y–2Y) i Stopa Fed',
        'Podaż pieniądza M2 (YoY %)',
        'Inflacja CPI vs Realna Stopa Fed',
        'Bezrobocie (Sahm Rule)',
        'VIX — Indeks Strachu',
        '',
        'Recession Scorecard — aktualny stan',
    ),
    vertical_spacing=0.10,
    horizontal_spacing=0.08,
    specs=[
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'indicator'}],
    ]
)

LEGEND_STYLE = dict(
    orientation='h',
    bgcolor='rgba(255,255,255,0.88)',
    bordercolor='#ccc',
    borderwidth=1,
    font=dict(size=10),
    xanchor='left',
    yanchor='top',
    tracegroupgap=0,
)

fig.update_layout(
    title=dict(
        text='<b>Fed, Pieniądz i Rynki</b> — Analiza Makroekonomiczna USA 1990–2026',
        font=dict(size=18)
    ),
    height=1600,
    template='plotly_white',
    font=dict(family='Inter, Arial, sans-serif', size=12),
    margin=dict(b=80),
    legend =dict(**LEGEND_STYLE, x=0.01, y=0.775),
    legend2=dict(**LEGEND_STYLE, x=0.55, y=0.775),
    legend3=dict(**LEGEND_STYLE, x=0.01, y=0.520),
    legend4=dict(**LEGEND_STYLE, x=0.55, y=0.520),
    legend5=dict(**LEGEND_STYLE, x=0.01, y=0.265),
    legend6=dict(**LEGEND_STYLE, x=0.55, y=0.265),
)

# ── PANEL 1: S&P500 ──────────────────────────────────────────────
sp = df.dropna(subset=['sp500'])
fig.add_trace(
    go.Scatter(
        x=sp['ym'], y=sp['sp500'],
        name='S&P500', legend='legend',
        line=dict(color=C_BLUE, width=1.5),
        hovertemplate='%{x|%Y-%m}: %{y:,.0f}<extra></extra>',
    ), row=1, col=1
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='markers',
               marker=dict(size=10, color='rgba(150,150,150,0.4)', symbol='square'),
               name='Okres recesji', legend='legend', showlegend=True),
    row=1, col=1
)
dodaj_recesje(fig, 1, 1, recesje_df)

# ── PANEL 2: Yield curve + Fed ───────────────────────────────────
yc = df.dropna(subset=['t10y2y'])
fig.add_trace(
    go.Scatter(
        x=yc['ym'], y=yc['t10y2y'],
        name='Yield Curve 10Y–2Y', legend='legend2',
        line=dict(color=C_RED, width=1.5),
        hovertemplate='%{x|%Y-%m}: %{y:.2f} pp<extra></extra>',
    ), row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=df['ym'], y=df['fedfunds'],
        name='Stopa Fed', legend='legend2',
        line=dict(color=C_ORANGE, width=1.5, dash='dot'),
        hovertemplate='%{x|%Y-%m}: %{y:.2f}%<extra></extra>',
    ), row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='lines',
               line=dict(color='#999', width=1, dash='dash'),
               name='Zero (inwersja poniżej)', legend='legend2', showlegend=True),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='markers',
               marker=dict(size=10, color='rgba(150,150,150,0.4)', symbol='square'),
               name='Okres recesji', legend='legend2', showlegend=True),
    row=1, col=2
)
fig.add_hline(y=0, line_dash='dash', line_color='#999', line_width=1, row=1, col=2)
dodaj_recesje(fig, 1, 2, recesje_df)

# ── PANEL 3: M2 YoY ──────────────────────────────────────────────
m2 = df.dropna(subset=['m2_yoy'])
fig.add_trace(
    go.Scatter(
        x=m2['ym'], y=m2['m2_yoy'],
        name='M2 YoY %', legend='legend3',
        fill='tozeroy', fillcolor='rgba(46,125,50,0.15)',
        line=dict(color=C_GREEN, width=1.5),
        hovertemplate='%{x|%Y-%m}: %{y:.1f}%<extra></extra>',
    ), row=2, col=1
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='markers',
               marker=dict(size=10, color='rgba(150,150,150,0.4)', symbol='square'),
               name='Okres recesji', legend='legend3', showlegend=True),
    row=2, col=1
)
fig.add_hline(y=0, line_dash='dash', line_color='#999', line_width=1, row=2, col=1)
dodaj_recesje(fig, 2, 1, recesje_df)

# ── PANEL 4: CPI YoY + realna stopa ──────────────────────────────
cpi = df.dropna(subset=['cpi_yoy'])
fig.add_trace(
    go.Scatter(
        x=cpi['ym'], y=cpi['cpi_yoy'],
        name='Inflacja CPI YoY %', legend='legend4',
        line=dict(color=C_RED, width=1.5),
        hovertemplate='%{x|%Y-%m}: %{y:.1f}%<extra></extra>',
    ), row=2, col=2
)
fig.add_trace(
    go.Scatter(
        x=df['ym'], y=df['real_rate'],
        name='Realna stopa Fed', legend='legend4',
        fill='tozeroy', fillcolor='rgba(101,31,165,0.10)',
        line=dict(color=C_PURPLE, width=1.5, dash='dot'),
        hovertemplate='%{x|%Y-%m}: %{y:.1f}%<extra></extra>',
    ), row=2, col=2
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='markers',
               marker=dict(size=10, color='rgba(150,150,150,0.4)', symbol='square'),
               name='Okres recesji', legend='legend4', showlegend=True),
    row=2, col=2
)
fig.add_hline(y=0, line_dash='dash', line_color='#999', line_width=1, row=2, col=2)
dodaj_recesje(fig, 2, 2, recesje_df)

# ── PANEL 5: Bezrobocie + Sahm Rule ──────────────────────────────
un = df.dropna(subset=['unrate'])
# min_periods toleruje pojedyncze braki w szeregu. Bez tego jeden pusty
# miesiac (UNRATE nie ma pazdziernika 2025) wycisza Sahm Rule na kolejne
# 12 miesiecy i zatrzymuje caly scorecard rok przed koncem danych.
df['unrate_3m']    = df['unrate'].rolling(3, min_periods=2).mean()
df['unrate_min12'] = df['unrate'].rolling(12, min_periods=10).min()
df['sahm']         = df['unrate_3m'] - df['unrate_min12']

fig.add_trace(
    go.Scatter(
        x=un['ym'], y=un['unrate'],
        name='Bezrobocie %', legend='legend5',
        line=dict(color=C_GRAY, width=1.5),
        hovertemplate='%{x|%Y-%m}: %{y:.1f}%<extra></extra>',
    ), row=3, col=1
)
sahm_data = df.dropna(subset=['sahm'])
fig.add_trace(
    go.Scatter(
        x=sahm_data['ym'], y=sahm_data['sahm'],
        name='Sahm Rule (odchylenie)', legend='legend5',
        line=dict(color=C_ORANGE, width=1.5, dash='dot'),
        hovertemplate='%{x|%Y-%m}: %{y:.2f}<extra></extra>',
    ), row=3, col=1
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='lines',
               line=dict(color=C_RED, width=1, dash='dash'),
               name='Próg Sahm Rule (0.5)', legend='legend5', showlegend=True),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='markers',
               marker=dict(size=10, color='rgba(150,150,150,0.4)', symbol='square'),
               name='Okres recesji', legend='legend5', showlegend=True),
    row=3, col=1
)
fig.add_hline(y=0.5, line_dash='dash', line_color=C_RED, line_width=1,
              annotation_text='Próg Sahm Rule (0.5)', annotation_position='top right',
              row=3, col=1)
dodaj_recesje(fig, 3, 1, recesje_df)

# ── PANEL 6: VIX — Indeks Strachu ────────────────────────────────
vix_data = df.dropna(subset=['vix'])
fig.add_trace(
    go.Scatter(
        x=vix_data['ym'], y=vix_data['vix'],
        name='VIX (śr. miesięczna)', legend='legend6',
        fill='tozeroy', fillcolor='rgba(230,81,0,0.10)',
        line=dict(color=C_ORANGE, width=1.5),
        hovertemplate='%{x|%Y-%m}: %{y:.1f}<extra></extra>',
    ), row=3, col=2
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='lines',
               line=dict(color='green', width=1, dash='dash'),
               name='Spokój (<15)', legend='legend6', showlegend=True),
    row=3, col=2
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='lines',
               line=dict(color='red', width=1, dash='dash'),
               name='Panika (>35)', legend='legend6', showlegend=True),
    row=3, col=2
)
fig.add_trace(
    go.Scatter(x=[None], y=[None], mode='markers',
               marker=dict(size=10, color='rgba(150,150,150,0.4)', symbol='square'),
               name='Okres recesji', legend='legend6', showlegend=True),
    row=3, col=2
)
fig.add_hline(y=15, line_dash='dash', line_color='green', line_width=1, opacity=0.5, row=3, col=2)
fig.add_hline(y=25, line_dash='dash', line_color='orange', line_width=1, opacity=0.5, row=3, col=2)
fig.add_hline(y=35, line_dash='dash', line_color='red', line_width=1, opacity=0.5, row=3, col=2)
dodaj_recesje(fig, 3, 2, recesje_df)

print('Wykresy skonfigurowane (8 paneli).')

Wykresy skonfigurowane (8 paneli).


In [9]:
# ── PANEL 8: Recession Scorecard (Indicator) — row 4 col 2 ───────
ostatni = df.dropna(subset=['fedfunds', 't10y2y', 'unrate', 'm2_yoy', 'sahm']).iloc[-1]

score = 0
szczegoly = []

# Yield curve
if ostatni['t10y2y'] < 0:
    score += 1
    szczegoly.append(('Yield Curve', f"{ostatni['t10y2y']:.2f} pp", '⚠️ Odwrócona'))
else:
    szczegoly.append(('Yield Curve', f"{ostatni['t10y2y']:.2f} pp", '✅ Normalna'))

# M2 YoY
if ostatni['m2_yoy'] < 0:
    score += 1
    szczegoly.append(('M2 YoY', f"{ostatni['m2_yoy']:.1f}%", '⚠️ Ujemny'))
else:
    szczegoly.append(('M2 YoY', f"{ostatni['m2_yoy']:.1f}%", '✅ Dodatni'))

# Realna stopa
real = ostatni['fedfunds'] - ostatni['cpi_yoy'] if not np.isnan(ostatni['cpi_yoy']) else np.nan
if not np.isnan(real) and real > 2:
    score += 1
    szczegoly.append(('Realna stopa Fed', f"{real:.1f}%", '⚠️ Restrykcyjna'))
else:
    szczegoly.append(('Realna stopa Fed', f"{real:.1f}%" if not np.isnan(real) else 'N/A', '✅ Neutralna'))

# Sahm Rule
if ostatni['sahm'] >= 0.5:
    score += 1
    szczegoly.append(('Sahm Rule', f"{ostatni['sahm']:.2f}", '🚨 ALARM (≥0.5)'))
elif ostatni['sahm'] >= 0.3:
    szczegoly.append(('Sahm Rule', f"{ostatni['sahm']:.2f}", '⚠️ Uwaga (≥0.3)'))
else:
    szczegoly.append(('Sahm Rule', f"{ostatni['sahm']:.2f}", '✅ OK (<0.3)'))

poziomy = {0: ('Niskie', '#2E7D32'), 1: ('Niskie', '#2E7D32'),
           2: ('Umiarkowane', '#E65100'), 3: ('Podwyższone', '#C62828'), 4: ('Wysokie', '#B71C1C')}
ryzyko_nazwa, ryzyko_kolor = poziomy[min(score, 4)]

fig.add_trace(
    go.Indicator(
        mode='gauge+number+delta',
        value=score,
        title=dict(text=f'Ryzyko recesji: <b>{ryzyko_nazwa}</b><br><span style="font-size:12px">({ostatni["ym"].strftime("%Y-%m")})</span>',
                   font=dict(size=16)),
        gauge=dict(
            axis=dict(range=[0, 4], tickvals=[0, 1, 2, 3, 4]),
            bar=dict(color=ryzyko_kolor),
            steps=[
                dict(range=[0, 1], color='#E8F5E9'),
                dict(range=[1, 2], color='#FFF9C4'),
                dict(range=[2, 3], color='#FFE0B2'),
                dict(range=[3, 4], color='#FFCDD2'),
            ],
            threshold=dict(line=dict(color='red', width=3), thickness=0.75, value=3)
        ),
        number=dict(font=dict(size=40, color=ryzyko_kolor)),
    ),
    row=4, col=2
)

print(f'Recession Scorecard: {score}/4 — {ryzyko_nazwa}')
print()
for wskaznik, wartosc, status in szczegoly:
    print(f'  {wskaznik:<20} {wartosc:>8}  {status}')

Recession Scorecard: 0/4 — Niskie

  Yield Curve           0.38 pp  ✅ Normalna
  M2 YoY                   5.4%  ✅ Dodatni
  Realna stopa Fed         0.3%  ✅ Neutralna
  Sahm Rule                0.10  ✅ OK (<0.3)


In [10]:
# Etykiety osi Y
fig.update_yaxes(title_text='Poziom indeksu', row=1, col=1)
fig.update_yaxes(title_text='Punkty procentowe', row=1, col=2)
fig.update_yaxes(title_text='% YoY', row=2, col=1)
fig.update_yaxes(title_text='%', row=2, col=2)
fig.update_yaxes(title_text='%', row=3, col=1)
fig.update_yaxes(title_text='VIX', row=3, col=2)

# Zapis do HTML
output_path = REP_DIR / 'dashboard.html'
fig.write_html(
    output_path,
    include_plotlyjs='cdn',
    full_html=True,
    config={'displayModeBar': True, 'scrollZoom': True}
)

print(f'Dashboard zapisany: {output_path.resolve()}')
print(f'Rozmiar pliku: {output_path.stat().st_size / 1024:.0f} KB')

fig.show()

Dashboard zapisany: /Users/wojciechpakulski/Documents/Wojtek/Projekty/Portfolio/FedPieniadzRynki/reports/dashboard.html
Rozmiar pliku: 180 KB


---
## Podsumowanie Dnia 3

In [11]:
conn.close()

print('DZIEŃ 3 UKOŃCZONY')
print()
print('Naprawy:')
print('  ✅ SP500 — pełna historia od 1990 (yfinance)')
print('  ✅ Q2 — inwersja → recesja (Python, wszystkie 3 recesje)')
print()
print('Nowe pliki:')
for p in sorted(REP_DIR.glob('*.html')):
    print(f'  {p.name} ({p.stat().st_size / 1024:.0f} KB)')
for p in sorted(RAW_DIR.glob('*full*')):
    print(f'  data/raw/{p.name}')
print()
print('Następny krok — Dzień 4:')
print('  Wskaźniki pochodne + percentyle historyczne w 03_analysis.ipynb')

DZIEŃ 3 UKOŃCZONY

Naprawy:
  ✅ SP500 — pełna historia od 1990 (yfinance)
  ✅ Q2 — inwersja → recesja (Python, wszystkie 3 recesje)

Nowe pliki:
  dashboard.html (180 KB)
  data/raw/SP500_full.csv

Następny krok — Dzień 4:
  Wskaźniki pochodne + percentyle historyczne w 03_analysis.ipynb
